In [1]:
import pandas as pd
import numpy as np
import pickle
import json

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

from xgboost import XGBRegressor


ModuleNotFoundError: No module named 'sklearn'

In [ ]:
df = pd.read_csv("irrigation_prediction.csv")

target = "Irrigation_Need"

X = df.drop(target, axis=1)
y = df[target]

# Identify columns
num_cols = X.select_dtypes(include=['int64','float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object']).columns.tolist()


In [ ]:
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
])

In [ ]:
models = {
    "linear": Pipeline([
        ("preprocess", preprocessor),
        ("model", LinearRegression())
    ]),
    
    "rf": Pipeline([
        ("preprocess", preprocessor),
        ("model", RandomForestRegressor(n_estimators=200, random_state=42))
    ]),
    
    "xgb": Pipeline([
        ("preprocess", preprocessor),
        ("model", XGBRegressor(n_estimators=300, learning_rate=0.05))
    ])
}

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [ ]:
for name, model in models.items():
    model.fit(X_train, y_train)
    print(f"{name} trained")


In [ ]:
for name, model in models.items():
    pickle.dump(model, open(f"{name}.pkl", "wb"))


In [ ]:
metadata = {
    "num_cols": num_cols,
    "cat_cols": cat_cols,
    "target_mean": float(y.mean())
}

json.dump(metadata, open("metadata.json", "w"))

print("✅ Models saved")